# 03 — LLM Fine-Tune (QLoRA) — Trace the Ace

Notebook C of the 4-notebook pipeline (see `Context.md`). Consumes
`trace-the-ace-raw` (transcripts + labels) and `cv_folds.csv` from
`trace-the-ace-features` (Notebook A's canonical split), and produces the
`trace-the-ace-llm` Kaggle Dataset consumed by Notebook D:

- `lora_adapter/fold_{i}/` — trained LoRA adapter + classification head per fold
- `oof_predictions_llm.csv` — out-of-fold predictions, columns `response_id, pred_llm`
- `llm_train_metrics.json` — per-fold + overall log loss / AUC, base-rate floor

## ⚠️ Manual setup required before running

**Set the accelerator to GPU T4 x2 with Internet ON** (Notebook settings panel,
right sidebar) before running any cell below. This notebook will not work on
CPU or with internet off, and 4-bit QLoRA needs the full 16GB of at least one
T4 (LoRA + gradient checkpointing keeps this fold's active model on a single
GPU; the second T4 is not required by the training code below, but leaving
both attached costs nothing extra against the GPU-hour budget beyond wall-clock
time, so multi-GPU parallelism across folds is a possible future optimization,
not something this notebook currently does).

## GPU-hour budget -- read before running

Context.md caps this project at **~30 GPU-hours/week** with a **12-hour
session cap**. These numbers are MEASURED (not estimated) from an actual
smoke-test run on a real Kaggle T4:

- **Training**: ~149s per optimizer step at effective batch size 16 (batch 2
  x grad-accum 8). A full fold's ~28,057 training responses would be ~1,754
  steps/epoch -> **~72.6 GPU-hours for one fold, one epoch** -- far beyond
  both the session cap and the entire weekly quota. `TRAIN_SAMPLE_SIZE`
  (section 3.3) subsamples each fold's training data down to a size that
  keeps training to roughly **~3 GPU-hours/fold** instead. This is a real
  data-efficiency trade-off (each fold's adapter sees less training signal),
  made under a hard 2-day deadline -- raise `TRAIN_SAMPLE_SIZE` if more
  budget becomes available, since OOF quality should improve monotonically
  with more training data.
- **Inference**: ~3.1s/example at batch size 2, and MEASURED (not assumed)
  that batch sizes 16, 8, and 4 all hit CUDA OOM on this T4 during actual
  inference -- a ~3000-token forward pass through a 7B 4-bit model leaves no
  real headroom above batch=2 even with trainer memory freed first, so
  inference gets no free batch-size win here. Since OOF coverage is needed
  across the FULL dataset (35,072 responses) regardless of training
  subsampling, scoring everyone would cost **~30 GPU-hours for inference
  alone** -- the entire weekly quota by itself. `VAL_SAMPLE_SIZE` (section
  3.3) subsamples each fold's held-out set too, cutting total inference to
  roughly **~6-7 GPU-hours across all 5 folds combined**. This is a real
  *response-level coverage* trade-off: `oof_predictions_llm.csv` will have
  gaps at the response_id level even once all 5 folds finish -- see section
  5 for how Notebook D must handle that.

**Net result**: expect roughly 3-4 GPU-hours per fold (training + that
fold's inference combined) with current settings -- meaning **up to 2
folds can plausibly fit in one 12-hour session**, and all 5 folds should fit
in ~15-20 GPU-hours total. With a tight multi-day deadline, run folds
back-to-back rather than spreading one per session:
1. Run the smoke test cell (section 3.4) first, in any session, to confirm the
   harness works end-to-end on real GPU hardware without spending meaningful
   quota (a few minutes, `FOLDS_TO_RUN=[0]` + `MAX_STEPS` capped very low).
2. Set `FOLDS_TO_RUN = [i, j]` to whichever 1-2 folds fit this session's
   remaining time budget, and run them to completion (or until the 12-hour
   cap).
3. Each fold checkpoints under `/kaggle/working/checkpoints/fold_{i}/` --
   re-running the same fold in a fresh session resumes from its last
   checkpoint automatically (section 3.3) rather than restarting.
4. If time runs out before all 5 folds finish, section 4's evaluation and
   section 5's outputs both handle a PARTIAL set of completed folds
   gracefully -- Notebook D can still stack whatever OOF predictions exist
   rather than getting nothing.


## 1. Setup

In [ ]:
# --- Installs --------------------------------------------------------------
# T4 = compute capability 7.5 (Turing): no native bf16 tensor-core support
# (that arrived with Ampere/sm_80+), so training runs in fp16 compute, and
# Flash-Attention-2 requires sm_80+ as well -- do NOT assume FA2 works here.
# Unsloth auto-detects compute capability and falls back to its own fused
# fp16-safe attention kernel (not FA2, not plain eager) on T4, so we let it
# choose rather than forcing an attn_implementation.
#
# IMPORTANT: do NOT let this install touch numpy's version AT ALL. Two
# earlier versions of this notebook learned this the hard way:
#   1. Hard-pinning old exact transformers/peft/bitsandbytes/trl versions
#      dragged numpy down to 1.26, breaking ABI compatibility with every
#      other package in the Kaggle image already compiled against numpy 2.x
#      (crashed `import pandas`).
#   2. Pinning only a numpy>=2 FLOOR (no ceiling) let pip upgrade numpy
#      2.0.2 -> 2.5.2 "to satisfy" unrelated packages -- that specific numpy
#      install came out internally broken (numpy._core.strings couldn't
#      import `_center` from numpy._core.umath, a numpy-vs-numpy problem,
#      not even a cross-package one).
# The fix: pin numpy to its EXACT pre-installed version so pip's resolver
# cannot change it in either direction, and let it fit everything else
# around that fixed point instead.
import numpy
_numpy_pin = numpy.__version__
print(f"pre-installed numpy: {_numpy_pin} (pinning resolver to numpy=={_numpy_pin}, exact)")


In [ ]:
# Single pip invocation so the resolver sees unsloth's own version ceilings
# on transformers/trl/datasets at the same time as everything else -- two
# separate pip calls let a later call silently install transformers/trl/
# datasets versions newer than what unsloth actually supports.
# --no-cache-dir + purging pip's cache afterward: a real production run once
# exhausted the container's disk mid-training (OSError: No space left on
# device); pip's own wheel cache for this large a dependency stack is one
# concrete, avoidable contributor to that.
!pip install -q -U --no-cache-dir "numpy=={_numpy_pin}" unsloth bitsandbytes peft transformers trl accelerate datasets
!pip cache purge


In [ ]:
# --- Disk usage diagnostics --------------------------------------------------
# Print free disk space now and again after model load / before each fold's
# training, so if disk pressure builds up we SEE it in logs well before a
# silent multi-hour crash, rather than after (which is what actually
# happened once: a run died at ~3h into fold 0 with zero prior warning).
import shutil as _shutil

def print_disk_usage(label):
    total, used, free = _shutil.disk_usage("/")
    print(f"[disk usage: {label}] total={total/1e9:.1f}GB used={used/1e9:.1f}GB free={free/1e9:.1f}GB")

def _dir_size_gb(path):
    p = Path(path) if not isinstance(path, Path) else path
    if not p.exists():
        return 0.0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

def print_cache_sizes(label):
    # Triton's JIT-compiled kernel cache is the prime suspect for unbounded
    # disk growth during training if batch shapes vary (see collate_fn's
    # fixed_length padding above) -- track it explicitly, not just total
    # disk usage, so a regression here is diagnosable rather than a mystery.
    triton_cache = Path(os.environ.get("TRITON_CACHE_DIR", Path.home() / ".triton" / "cache"))
    hf_cache = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
    print(f"[cache sizes: {label}] triton={_dir_size_gb(triton_cache):.2f}GB "
          f"hf_hub={_dir_size_gb(hf_cache):.2f}GB "
          f"checkpoints={_dir_size_gb(CHECKPOINT_ROOT) if 'CHECKPOINT_ROOT' in globals() else 0:.2f}GB")

print_disk_usage("after installs")


In [ ]:
import os

# IMPORTANT: force single-GPU visibility BEFORE torch ever touches CUDA.
# Kaggle's "T4 x2" gives 2 physical GPUs, and HF Trainer auto-wraps the model
# in torch.nn.DataParallel whenever torch.cuda.device_count() > 1 with no
# explicit distributed setup. Unsloth's fused/quantized LoRA kernels are NOT
# DataParallel-safe -- naive DataParallel replication corrupts the packed
# 4-bit weight tensors, which surfaced as a shape-mismatch RuntimeError deep
# inside Unsloth's matmul_lora path during an actual training step (not at
# model-load time, which is why this only shows up once training starts).
# This notebook is single-GPU by design (see the GPU-hour usage note above);
# the second T4 sits idle, which is fine given per-fold checkpointing already
# bounds a session to one fold at a time.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re
import gc
import json
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import log_loss, roc_auc_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_FOLDS = 5

# --- Confirm we're actually on a T4 (or at least a supported 4-bit GPU) -----
assert torch.cuda.is_available(), "No GPU detected -- set the accelerator to GPU T4 x2 in notebook settings"
device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"GPU: {device_name}, compute capability: {capability}")
if capability < (7, 5):
    print("WARNING: compute capability below 7.5 (T4) -- 4-bit NF4 QLoRA may not be supported.")
if capability < (8, 0):
    print("NOTE: compute capability below 8.0 -- Flash-Attention-2 and bf16 tensor cores are "
          "NOT available; Unsloth will fall back to its fp16-safe attention path automatically.")


In [ ]:
def _resolve_dataset_dir(slug: str, sentinel_file: str) -> Path:
    """Kaggle mounts a dataset at /kaggle/input/<slug>/ normally, but some
    environments nest it under /kaggle/input/datasets/<owner>/<slug>/ instead
    (observed in Notebook A) -- auto-detect whichever is present."""
    direct = Path("/kaggle/input") / slug
    if (direct / sentinel_file).exists():
        return direct
    nested_root = Path("/kaggle/input/datasets")
    if nested_root.exists():
        for owner_dir in nested_root.iterdir():
            candidate = owner_dir / slug
            if (candidate / sentinel_file).exists():
                return candidate
    raise FileNotFoundError(
        f"Could not locate '{slug}' dataset under /kaggle/input "
        f"(checked {direct} and /kaggle/input/datasets/*/{slug})"
    )

RAW_DIR = _resolve_dataset_dir("trace-the-ace-raw", "train_features.csv")
TRANSCRIPTS_DIR = RAW_DIR / "train_transcripts"
FEATURES_DIR = _resolve_dataset_dir("trace-the-ace-features", "cv_folds.csv")
OUT_DIR = Path("/kaggle/working")
CHECKPOINT_ROOT = OUT_DIR / "checkpoints"
ADAPTER_ROOT = OUT_DIR / "lora_adapter"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)

print(f"RAW_DIR:      {RAW_DIR}")
print(f"FEATURES_DIR: {FEATURES_DIR}")


In [ ]:
# --- Restore state from a previous version of THIS SAME kernel, if attached --
# /kaggle/working/ does NOT persist between separate kernel executions on its
# own -- running fold 2/3 in a fresh session would otherwise see none of fold
# 0/1's checkpoints, adapters, or partial OOF predictions. The fix: add this
# kernel's own slug to kernel_sources in kernel-metadata.json (self-reference
# -- Kaggle resolves it to the latest COMPLETED prior version's output, not
# the in-progress one), which mounts that prior output here. Copy forward
# anything found before running more folds.
import shutil

_prev_dirs = [Path("/kaggle/input/03-llm-finetune")]
_nested = Path("/kaggle/input/datasets")
if _nested.exists():
    for owner_dir in _nested.iterdir():
        cand = owner_dir / "03-llm-finetune"
        if cand.exists():
            _prev_dirs.append(cand)

for _prev_dir in _prev_dirs:
    if not _prev_dir.exists():
        continue
    for _sub in ["checkpoints", "lora_adapter"]:
        _src = _prev_dir / _sub
        if not _src.exists():
            continue
        for _item in _src.rglob("*"):
            _rel = _item.relative_to(_src)
            _target = (OUT_DIR / _sub) / _rel
            if _item.is_dir():
                _target.mkdir(parents=True, exist_ok=True)
            elif not _target.exists():
                _target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(_item, _target)
    _prev_partial = _prev_dir / "oof_predictions_llm_partial.csv"
    _cur_partial = OUT_DIR / "oof_predictions_llm_partial.csv"
    if _prev_partial.exists() and not _cur_partial.exists():
        shutil.copy2(_prev_partial, _cur_partial)
    print(f"Restored prior kernel state from {_prev_dir}")


In [ ]:
# --- Hugging Face auth (avoids rate limits / unauthenticated-request warnings
# when downloading the base model) -----------------------------------------
from huggingface_hub import login as hf_login

try:
    hf_token_dir = _resolve_dataset_dir("hf-token", "hf_token.txt")
    hf_token = (hf_token_dir / "hf_token.txt").read_text().strip()
    os.environ["HF_TOKEN"] = hf_token
    hf_login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub")
except Exception as e:
    print(f"Could not load HF token ({e!r}) -- continuing with unauthenticated HF requests")


In [ ]:
# --- Load response-level data + reuse Notebook A's canonical fold split ------
train_features = pd.read_csv(RAW_DIR / "train_features.csv")
train_labels = pd.read_csv(RAW_DIR / "train_labels.csv")
cv_folds = pd.read_csv(FEATURES_DIR / "cv_folds.csv")

df = train_features.merge(train_labels, on="response_id", how="inner", validate="one_to_one")
df = df.merge(cv_folds, on="session_id", how="left")
assert df["fold"].isna().sum() == 0, "every response must have a non-null fold assignment"
df["fold"] = df["fold"].astype(int)

print(df.shape)
print(df["fold"].value_counts().sort_index())
print(f"base rate is_correct: {df['is_correct'].mean():.4f}")


In [ ]:
# --- Load every transcript once (same pattern as Notebook A) ----------------
session_ids = df["session_id"].unique().tolist()

def load_transcript(session_id: str) -> pd.DataFrame:
    t = pd.read_csv(TRANSCRIPTS_DIR / f"{session_id}.csv")
    return t.sort_values("utterance_id").reset_index(drop=True)

def _load(sid):
    return sid, load_transcript(sid)

transcripts = {}
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = [ex.submit(_load, sid) for sid in session_ids]
    for fut in as_completed(futures):
        sid, t = fut.result()
        transcripts[sid] = t

print(f"loaded {len(transcripts)} transcripts")


In [ ]:
# --- Load base model in 4-bit via Unsloth -----------------------------------
# Qwen2.5-7B-Instruct (Apache-2.0, commercially usable) is the preferred base
# per Context.md sec 4. Fall back to Llama-3.1-8B-Instruct ONLY if it fails to
# load -- that fallback carries the Llama 3.1 Community License, which has its
# own commercial-use terms; re-check those terms before relying on it for a
# prize-eligible final submission.
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 3072
# Notebook A's EDA: median session is ~267 turns / ~913 student words; the
# 75th percentile is ~316 turns / ~1203 student words. Full turn text (tutor +
# student combined) runs noticeably higher than student-word-count alone, so
# 3072 tokens comfortably covers roughly the median-to-75th-percentile session
# in full, while still bounding VRAM/compute for the long tail (max observed:
# 622 turns / 3350 student words) via the truncation strategy in section 2.
# This also leaves headroom on a 16GB T4 for LoRA optimizer states + gradient
# checkpointing + a batch size >= 2 in 4-bit.

BASE_MODEL_CANDIDATES = [
    ("Qwen/Qwen2.5-7B-Instruct", "Apache-2.0"),
    ("meta-llama/Llama-3.1-8B-Instruct", "Llama 3.1 Community License"),
]

base_model = None
tokenizer = None
loaded_model_name = None
for model_name, license_name in BASE_MODEL_CANDIDATES:
    try:
        base_model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=MAX_SEQ_LEN,
            dtype=None,  # auto -- resolves to fp16 on T4 (no bf16 tensor cores pre-Ampere)
            load_in_4bit=True,
        )
        loaded_model_name = model_name
        if model_name != BASE_MODEL_CANDIDATES[0][0]:
            print(f"WARNING: fell back to {model_name} ({license_name}). "
                  f"Re-check commercial-use terms before a prize-eligible submission.")
        break
    except Exception as e:
        print(f"Failed to load {model_name}: {e!r}")

assert base_model is not None, "Both base model candidates failed to load"
print(f"Loaded base model: {loaded_model_name}")
print_disk_usage("after model load")
print_cache_sizes("after model load")


## 2. Prompt / Input Formatting

In [ ]:
# --- Session -> prompt body, with truncation for long sessions ---------------
# Strategy (Context.md sec 8 "context-window truncation" open decision,
# resolved here): keep the first HEAD_TURNS turns verbatim (session opening --
# frequently states context/objective framing), keep as much of the tail as
# fits (closest to the implied assessment, most temporally relevant to the
# knowledge-tracing target per the problem statement's "preserve sequence
# information" guidance), and drop from the MIDDLE first when a session
# exceeds budget, replacing the dropped span with an explicit marker so the
# model knows a gap exists rather than silently seeing a discontinuity.
HEAD_TURNS = 5
RESERVED_OVERHEAD_TOKENS = 150  # preamble + objective text + suffix + safety margin

SYSTEM_PREAMBLE = (
    "You are evaluating a tutoring session transcript to predict whether the "
    "student will answer the next assessment question correctly.\n\n"
    "Learning objective being assessed: {objective}\n\n"
    "Tutoring session transcript:\n"
)
FINAL_INSTRUCTION = (
    "\n\nBased on this tutoring session, will the student answer the next "
    "assessment question on this objective correctly?"
)

def _format_turn(role: str, content: str) -> str:
    speaker = "Tutor" if role == "tutor" else "Student"
    return f"{speaker}: {content.strip()}"

def _n_tokens(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def build_session_body(transcript_df: pd.DataFrame, max_tokens: int = MAX_SEQ_LEN,
                        head_turns: int = HEAD_TURNS,
                        reserved_overhead: int = RESERVED_OVERHEAD_TOKENS) -> str:
    """Build the (possibly truncated) transcript body for one session. Cached
    per session_id since the same body is reused across every response that
    shares this session (only the learning_objective differs per response)."""
    turns = transcript_df[transcript_df["role"].isin(["tutor", "student"])]
    turn_lines = [
        _format_turn(row.role, row.content)
        for row in turns.itertuples()
        if isinstance(row.content, str) and row.content.strip()
    ]
    if not turn_lines:
        return "(no transcript content available)"

    budget = max_tokens - reserved_overhead
    full_text = "\n".join(turn_lines)
    if _n_tokens(full_text) <= budget:
        return full_text

    head_lines = turn_lines[:head_turns]
    head_text = "\n".join(head_lines)
    head_len = _n_tokens(head_text)
    remaining = max(budget - head_len - 20, 0)  # 20 tokens reserved for the marker line

    tail_lines = []
    tail_len = 0
    for line in reversed(turn_lines[head_turns:]):
        line_len = _n_tokens(line)
        if tail_len + line_len > remaining:
            break
        tail_lines.insert(0, line)
        tail_len += line_len

    n_omitted = len(turn_lines) - head_turns - len(tail_lines)
    parts = [head_text]
    if n_omitted > 0:
        parts.append(f"[... {n_omitted} turns omitted ...]")
    if tail_lines:
        parts.append("\n".join(tail_lines))
    return "\n".join(p for p in parts if p)

def build_prompt(transcript_df: pd.DataFrame, learning_objective: str,
                  max_tokens: int = MAX_SEQ_LEN, head_turns: int = HEAD_TURNS) -> str:
    """Public entry point matching the spec: one session's transcript + its
    learning_objective -> one prompt string."""
    body = build_session_body(transcript_df, max_tokens=max_tokens, head_turns=head_turns)
    return SYSTEM_PREAMBLE.format(objective=learning_objective) + body + FINAL_INSTRUCTION


In [ ]:
# --- Cache the truncated body once per session, build final prompts per response ---
session_bodies = {}
for sid in session_ids:
    session_bodies[sid] = build_session_body(transcripts[sid])

df["prompt_text"] = [
    SYSTEM_PREAMBLE.format(objective=obj) + session_bodies[sid] + FINAL_INSTRUCTION
    for sid, obj in zip(df["session_id"], df["learning_objective"])
]

sample_lengths = df["prompt_text"].sample(min(500, len(df)), random_state=RANDOM_STATE).map(_n_tokens)
print(sample_lengths.describe())
print(f"\n--- example prompt (truncated to first 800 chars for display) ---\n")
print(df['prompt_text'].iloc[0][:800])


### Target framing: classification head vs. constrained generation

Two established styles for turning an LLM into a binary-correctness predictor:

- **LLMKT-style (constrained generation)**: fine-tune the causal LM to emit a
  literal token (e.g. "Yes"/"No") as its next-token prediction, then read the
  predicted probability off the softmax restricted to those candidate tokens.
- **CLST-style (classification head)**: attach a small linear head to the
  final hidden state of the last input token and train it directly with
  binary cross-entropy.

**We use the classification-head approach.** Reasons:
1. It optimizes binary cross-entropy directly -- exactly the competition
   metric (log loss) -- rather than a generic next-token softmax over the
   full vocabulary that happens to get restricted to two tokens post hoc.
2. It sidesteps tokenizer-specific issues where "correct"/"incorrect" (or
   "Yes"/"No") may not be single, stable tokens across model vocabularies.
3. The raw logit is a plain linear function of the pooled hidden state, which
   is simple to temperature-scale/calibrate downstream in Notebook D if
   needed.

The trade-off: this approach can't be prompted/inspected as naturally as a
generative one (there's no literal text to read at inference time, just a
scalar), but since the only artifact Notebook D needs is a calibrated
probability per `response_id`, that's not a cost here.


## 3. Training (QLoRA / PEFT)

In [ ]:
# --- Classification head wrapper --------------------------------------------
# Pools the last non-padded token's final hidden state and projects it to a
# single logit. Trained jointly with the LoRA adapter via BCEWithLogitsLoss --
# NOT the base model's generic language-modeling loss.
class SessionCorrectnessClassifier(nn.Module):
    def __init__(self, peft_model, hidden_size: int):
        super().__init__()
        self.peft_model = peft_model
        self.classifier_head = nn.Linear(hidden_size, 1)
        nn.init.zeros_(self.classifier_head.bias)
        nn.init.normal_(self.classifier_head.weight, std=0.02)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.peft_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )
        hidden = outputs.hidden_states[-1]  # (batch, seq, hidden_size)
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = hidden[torch.arange(hidden.size(0), device=hidden.device), last_idx]
        logits = self.classifier_head(pooled.to(self.classifier_head.weight.dtype)).squeeze(-1)

        loss = None
        if labels is not None:
            loss = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
        return {"loss": loss, "logits": logits}


In [ ]:
# --- Dataset + collator ------------------------------------------------------
from torch.utils.data import Dataset

class PromptDataset(Dataset):
    def __init__(self, prompts, labels, tokenizer, max_length=MAX_SEQ_LEN):
        self.prompts = list(prompts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.prompts[idx], truncation=True, max_length=self.max_length,
            add_special_tokens=True,
        )
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": float(self.labels[idx]),
        }

def collate_fn(batch, tokenizer, fixed_length=None):
    # Pad to a FIXED length (MAX_SEQ_LEN) rather than each batch's own max --
    # dynamic per-batch padding produces a new tensor shape almost every
    # step (real prompt lengths vary batch to batch), and Unsloth/Triton's
    # JIT-compiled kernels cache a distinct compiled variant PER SHAPE seen.
    # This is the prime suspect behind a real production run silently
    # exhausting disk mid-training (kernel cache growth with no warning)
    # after ~50-70 steps -- fixed-length padding bounds the shape space to
    # exactly one, so at most a handful of kernel variants ever get compiled
    # and cached. Costs a little wasted compute on padding tokens, which is
    # a small trade here since observed prompt lengths already cluster near
    # MAX_SEQ_LEN (mean ~2937 of 3072 in the smoke test).
    max_len = fixed_length if fixed_length is not None else max(len(b["input_ids"]) for b in batch)
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    input_ids, attn, labels = [], [], []
    for b in batch:
        n_pad = max_len - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad_id] * n_pad)
        attn.append(b["attention_mask"] + [0] * n_pad)
        labels.append(b["labels"])
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.float),
    }


In [ ]:
# --- LoRA config + custom BCE Trainer ----------------------------------------
from transformers import Trainer, TrainingArguments

LORA_CONFIG = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_STATE,
)

# NOTE: Trainer-level mid-training checkpointing (save_steps) is DISABLED
# below (save_strategy="no"). MEASURED root cause of the real production
# failure: a Trainer checkpoint of this wrapped model (SessionCorrectness
# Classifier is a plain nn.Module, not a recognized PreTrainedModel, so
# Trainer's default save path doesn't know to save only the LoRA deltas) hit
# 7.63GB after just 5 training steps in a follow-up smoke test -- it was
# very likely serializing the entire frozen 7B base alongside the adapter.
# With save_steps firing repeatedly across a real ~75-step fold, this is
# what silently exhausted the container's disk mid-training. The small,
# CORRECT save already happens manually after training via
# `peft_model.save_pretrained()` (adapter deltas only, a few hundred MB) --
# that's sufficient given a fold now only takes ~3 GPU-hours; if a session
# times out mid-fold, the fold restarts from scratch rather than resuming
# from a checkpoint, which is an acceptable trade against the disk risk.
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8     # effective batch size 16
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1

# --- Training-set subsampling (measured, not estimated) ---------------------
# Smoke-test measurement on real T4 hardware: ~149s per optimizer step at
# effective batch size 16. A full fold's ~28,057 training responses would be
# ~1,754 steps/epoch -> ~72.6 GPU-hours for ONE fold, ONE epoch -- far beyond
# both the 12h session cap and the entire weekly 30h quota, so TRAIN_SAMPLE_SIZE
# subsamples (stratified by is_correct, a different draw per fold via the
# fold-offset random seed) down to a tractable size.
#
# DIAGNOSTIC NOTE: a first real run at TRAIN_SAMPLE_SIZE=1200 (75 steps, 1
# epoch) produced a classifier that collapsed to near-constant high-confidence
# majority-class predictions -- AUC ~0.51 (barely above the 0.50 random
# floor) and log loss WORSE than the trivial base-rate constant prediction.
# That's a classic underfitting signature: 75 gradient steps is very little
# signal for a randomly-initialized classification head + LoRA adapters to
# learn a subtle transcript-to-correctness pattern from noisy conversational
# text. This value is bumped up (~3.3x, ~250 steps) specifically to test
# whether more training data/steps fixes the collapse before committing
# further budget to redoing all 5 folds at this larger size.
TRAIN_SAMPLE_SIZE = 4000

# --- Inference batching ------------------------------------------------------
# MEASURED (not assumed): batch sizes 16, 8, and 4 all hit CUDA OOM on this
# T4 during actual inference (auto-halving retry confirmed each one fails in
# turn) -- a ~3000-token forward pass through a 7B 4-bit model apparently
# leaves no real headroom above batch=2 even with trainer memory freed
# first. So inference does NOT get a free batch-size win the way it would on
# a bigger GPU; measured throughput stays ~3.1s/example. Going straight to
# batch=2 avoids wasting ~1-2 minutes per fold on failed OOM retries.
INFERENCE_BATCH_SIZE = 2

# --- Validation-set subsampling for inference (coverage trade-off) ----------
# At ~3.1s/example, scoring every response across all 5 folds (the full
# 35,072-response dataset -- inference cost does NOT shrink with
# TRAIN_SAMPLE_SIZE, since OOF coverage is a separate requirement from
# training data volume) would cost ~30 GPU-hours by itself, on top of
# training. Given a hard 2-day deadline, VAL_SAMPLE_SIZE subsamples each
# fold's held-out set (stratified by is_correct) down to a size that keeps
# total inference to roughly 6-7 GPU-hours across all 5 folds combined. This
# is a genuine coverage trade-off: oof_predictions_llm.csv will NOT have a
# row for every response_id in the full dataset -- see section 5's markdown
# for how Notebook D should handle the gap (fall back to pred_baseline for
# response_ids missing an LLM prediction). Raise this (or remove the cap
# entirely) if more GPU-hour budget becomes available later.
VAL_SAMPLE_SIZE = 1500

class BCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

def fresh_peft_classifier(base_model):
    """Wrap the (shared, frozen) 4-bit base model with a NEW LoRA adapter +
    classification head for one fold. Call `.peft_model.unload()` on the
    result's underlying peft model when done with a fold to reset the base
    model to its clean state before the next fold, rather than reloading the
    7B weights from disk each time (saves real GPU-hours across 5 folds)."""
    peft_model = FastLanguageModel.get_peft_model(base_model, **LORA_CONFIG)
    hidden_size = peft_model.config.hidden_size
    return SessionCorrectnessClassifier(peft_model, hidden_size)


### Resuming across sessions (fold-level, not step-level)

Trainer-level mid-fold checkpointing is disabled (see the disk-footprint
note in the previous cell -- it was the measured cause of a real
disk-exhaustion crash). Resumability is now at the FOLD level, not the
training-step level: if a Kaggle session times out mid-fold, that fold's
training restarts from scratch in the next session rather than resuming from
a partial checkpoint. Given a fold now only costs ~3 GPU-hours, this is a
deliberate, acceptable trade against the disk risk. Completed folds'
adapters + OOF predictions ARE carried forward across sessions via the
self-referencing `kernel_sources` restore logic above -- only an
in-progress, not-yet-finished fold loses its partial progress.


In [ ]:
def run_fold(fold_idx: int, base_model, tokenizer, df: pd.DataFrame,
             max_steps: int = None) -> pd.DataFrame:
    """Train on all folds except fold_idx, predict on fold_idx, save the
    adapter + classifier head, and return this fold's OOF predictions
    immediately (so partial progress survives a session timeout)."""
    fold_dir = CHECKPOINT_ROOT / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    adapter_dir = ADAPTER_ROOT / f"fold_{fold_idx}"

    print_disk_usage(f"start of fold {fold_idx}")
    print_cache_sizes(f"start of fold {fold_idx}")

    train_df = df[df["fold"] != fold_idx].reset_index(drop=True)
    val_df = df[df["fold"] == fold_idx].reset_index(drop=True)
    print(f"[fold {fold_idx}] full train={len(train_df)} val={len(val_df)}")

    def _stratified_subsample(frame, target_size, seed):
        """Explicit boolean-mask stratification, not groupby().apply() --
        the latter silently drops the grouping column (is_correct) on recent
        pandas versions, which would break every downstream reference to it."""
        frac = target_size / len(frame)
        pos = frame[frame["is_correct"] == 1].sample(frac=frac, random_state=seed)
        neg = frame[frame["is_correct"] == 0].sample(frac=frac, random_state=seed)
        return pd.concat([pos, neg], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

    # Subsample training data (see TRAIN_SAMPLE_SIZE comment above) --
    # stratified by is_correct so the subsample's base rate matches the
    # full fold's, with a fold-specific seed so each fold's adapter trains
    # on a different slice rather than identical rows.
    if TRAIN_SAMPLE_SIZE is not None and len(train_df) > TRAIN_SAMPLE_SIZE:
        train_df = _stratified_subsample(train_df, TRAIN_SAMPLE_SIZE, RANDOM_STATE + fold_idx)
        print(f"[fold {fold_idx}] subsampled train={len(train_df)} (base rate {train_df['is_correct'].mean():.3f})")

    # Subsample the validation/inference set too (see VAL_SAMPLE_SIZE comment
    # above) -- a real coverage trade-off under the time budget: responses
    # NOT in this subsample get no row in oof_predictions_llm.csv at all, so
    # Notebook D must fall back to pred_baseline for those response_ids.
    full_val_df = val_df
    if VAL_SAMPLE_SIZE is not None and len(val_df) > VAL_SAMPLE_SIZE:
        val_df = _stratified_subsample(val_df, VAL_SAMPLE_SIZE, RANDOM_STATE + fold_idx)
        print(f"[fold {fold_idx}] subsampled val={len(val_df)} of {len(full_val_df)} "
              f"(base rate {val_df['is_correct'].mean():.3f}) -- "
              f"{len(full_val_df) - len(val_df)} responses in this fold will have NO pred_llm")

    model = fresh_peft_classifier(base_model)

    train_ds = PromptDataset(train_df["prompt_text"], train_df["is_correct"], tokenizer)
    val_ds = PromptDataset(val_df["prompt_text"], val_df["is_correct"], tokenizer)

    args = TrainingArguments(
        output_dir=str(fold_dir),
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        max_steps=max_steps if max_steps is not None else -1,
        save_strategy="no",  # see the disk-footprint note above -- do NOT enable step-based checkpointing
        logging_steps=5,
        fp16=True,   # T4 has no bf16 tensor cores -- fp16 compute only
        bf16=False,
        report_to=[],
        remove_unused_columns=False,
        dataloader_num_workers=2,
    )

    trainer = BCETrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        data_collator=lambda batch: collate_fn(batch, tokenizer, fixed_length=MAX_SEQ_LEN),
    )

    trainer.train()
    print_disk_usage(f"fold {fold_idx} after training")
    print_cache_sizes(f"fold {fold_idx} after training")

    # --- Save adapter + classifier head for this fold ---
    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.peft_model.save_pretrained(str(adapter_dir))
    torch.save(model.classifier_head.state_dict(), adapter_dir / "classifier_head.pt")
    print(f"[fold {fold_idx}] saved adapter to {adapter_dir}")

    # --- Free training-time GPU memory (optimizer states, gradients, the
    # Trainer's internal references) BEFORE inference. Training runs right up
    # against the T4's 16GB (observed ~13.2/14.56 GiB in use at batch_size=2,
    # seq_len~3000), so anything left allocated from training pushes
    # inference straight into an OOM -- this is what actually happened in an
    # earlier version of this notebook: training completed fine, then
    # inference (which also used a LARGER batch size than training) OOM'd
    # immediately after.
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

    # --- Inference on the held-out fold, immediately (bounds memory + gives
    # usable partial OOF predictions even if the notebook times out later) ---
    # Uses a LARGER batch than training (no backward pass / optimizer state
    # competing for VRAM once trainer memory is freed above) since OOF
    # predictions are needed for every response across all 5 folds, not just
    # a subsample -- inference cost doesn't shrink with TRAIN_SAMPLE_SIZE, so
    # this is the more important place to optimize for the overall time
    # budget. INFERENCE_BATCH_SIZE is a starting guess, not a measured-safe
    # value; auto-halve on OOM rather than losing the whole fold's run.
    FastLanguageModel.for_inference(model.peft_model)
    model.eval()
    val_preds = []
    infer_batch_size = INFERENCE_BATCH_SIZE
    with torch.no_grad():
        i = 0
        while i < len(val_ds):
            end = min(i + infer_batch_size, len(val_ds))
            batch = [val_ds[j] for j in range(i, end)]
            batch = collate_fn(batch, tokenizer, fixed_length=MAX_SEQ_LEN)
            batch = {k: v.to(model.classifier_head.weight.device) for k, v in batch.items() if k != "labels"}
            try:
                logits = model(**batch)["logits"]
                val_preds.extend(torch.sigmoid(logits).float().cpu().numpy().tolist())
                i = end
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if infer_batch_size <= 1:
                    raise
                infer_batch_size = max(1, infer_batch_size // 2)
                print(f"[fold {fold_idx}] inference OOM at batch={infer_batch_size * 2}, "
                      f"halving to {infer_batch_size} and retrying this batch")

    fold_oof = pd.DataFrame({
        "response_id": val_df["response_id"].values,
        "pred_llm": val_preds,
        "fold": fold_idx,
    })

    # Reset the shared base model to clean state for the next fold, avoiding
    # a full reload of the 7B weights from disk.
    base_model = model.peft_model.unload()
    del model
    torch.cuda.empty_cache()

    return fold_oof


### 3.4 Smoke test (run this first, in any session)

Trains fold 0 for a tiny number of steps on a small subsample, purely to
confirm the whole harness (data loading, tokenization, LoRA + classifier
head, checkpointing, inference, saving) runs without error on real GPU
hardware. This should finish in a few minutes and costs negligible GPU-hour
budget -- do this before committing a full session to real training.


In [ ]:
SMOKE_TEST = False  # flip to True to re-verify the harness before a real run

if SMOKE_TEST:
    smoke_df = df.sample(n=min(200, len(df)), random_state=RANDOM_STATE).copy()
    # Ensure both classes and both train/val sides of fold 0 are represented.
    if smoke_df["fold"].nunique() < 2 or smoke_df["is_correct"].nunique() < 2:
        # Explicit boolean-mask sampling, not groupby().apply() -- the latter
        # silently drops the grouping columns on recent pandas versions,
        # which would break run_fold's own df["fold"]/df["is_correct"] use.
        parts = []
        for f in df["fold"].unique():
            for label in df["is_correct"].unique():
                subset = df[(df["fold"] == f) & (df["is_correct"] == label)]
                parts.append(subset.sample(min(20, len(subset)), random_state=RANDOM_STATE))
        smoke_df = pd.concat(parts, ignore_index=True)
    smoke_oof = run_fold(0, base_model, tokenizer, smoke_df, max_steps=5)
    print(smoke_oof.head())
    print("Smoke test passed -- harness runs end-to-end.")


## 3.5 Full fold training

**Parameterize which fold(s) to run in this Kaggle session here.**
Currently set to fold 0 ONLY -- this is a deliberate diagnostic run at the
larger `TRAIN_SAMPLE_SIZE` (see the note above) to confirm more training
data actually fixes the underfitting collapse before spending budget
redoing all 5 folds at this size. Fold 1's earlier (smaller-sample) results
are preserved via the self-referencing `kernel_sources` restore and will
show up alongside fold 0's new result in section 4's evaluation, giving a
direct before/after comparison on the same run.


In [ ]:
FOLDS_TO_RUN = [0]  # diagnostic: fold 0 only, at the larger TRAIN_SAMPLE_SIZE above

all_fold_oof = []
for fold_idx in FOLDS_TO_RUN:
    fold_oof = run_fold(fold_idx, base_model, tokenizer, df)
    all_fold_oof.append(fold_oof)
    # Persist partial OOF predictions after every fold so a mid-run timeout
    # still leaves usable output on disk.
    partial_path = OUT_DIR / "oof_predictions_llm_partial.csv"
    existing = []
    if partial_path.exists():
        existing_df = pd.read_csv(partial_path)
        existing = [existing_df[existing_df["fold"] != fold_idx]]
    pd.concat(existing + [fold_oof], ignore_index=True).to_csv(partial_path, index=False)
    print(f"[fold {fold_idx}] partial OOF predictions saved to {partial_path}")


## 4. Evaluation

Metrics below are computed on whichever folds have been run **so far** across
however many Kaggle sessions this notebook has been executed in (loaded from
the accumulated `oof_predictions_llm_partial.csv`). Once all 5 folds are
present, this doubles as the final evaluation for section 5's outputs.

**Two separate coverage dimensions, both partial under the current time
budget**: (1) *fold* coverage -- some folds may not have run yet in this
session; (2) *response* coverage within a completed fold -- `VAL_SAMPLE_SIZE`
means only a stratified subsample of each fold's held-out responses actually
got scored (see section 3.3). Metrics below reflect exactly what was scored,
which is a valid (if smaller) OOF estimate; it is section 5's
`oof_predictions_llm.csv` that will have gaps at the response_id level for
Notebook D to handle.


In [ ]:
oof_path = OUT_DIR / "oof_predictions_llm_partial.csv"
assert oof_path.exists(), "run at least one fold in section 3.5 before evaluating"
oof_all = pd.read_csv(oof_path)

folds_done = sorted(oof_all["fold"].unique().tolist())
print(f"Folds completed so far: {folds_done} (of {list(range(N_FOLDS))})")
if len(folds_done) < N_FOLDS:
    print("NOTE: not all folds are done yet -- metrics below are PARTIAL, "
          "computed only over the completed folds' held-out responses.")

eval_df = oof_all.merge(df[["response_id", "is_correct"]], on="response_id", how="left")

per_fold_metrics = []
for fold_idx in folds_done:
    fold_rows = eval_df[eval_df["fold"] == fold_idx]
    fold_log_loss = log_loss(fold_rows["is_correct"], fold_rows["pred_llm"], labels=[0, 1])
    fold_auc = roc_auc_score(fold_rows["is_correct"], fold_rows["pred_llm"])

    train_rows = df[df["fold"] != fold_idx]
    base_rate = train_rows["is_correct"].mean()
    base_rate_pred = np.full(len(fold_rows), base_rate)
    base_rate_log_loss = log_loss(fold_rows["is_correct"], base_rate_pred, labels=[0, 1])
    base_rate_auc = roc_auc_score(fold_rows["is_correct"], base_rate_pred)

    n_val_full = int((df["fold"] == fold_idx).sum())
    per_fold_metrics.append({
        "fold": fold_idx, "n_val": len(fold_rows), "n_val_full": n_val_full,
        "log_loss": fold_log_loss, "auc": fold_auc,
        "base_rate_log_loss": base_rate_log_loss, "base_rate_auc": base_rate_auc,
    })
    print(f"fold {fold_idx}: log_loss={fold_log_loss:.4f} auc={fold_auc:.4f} "
          f"| base-rate floor log_loss={base_rate_log_loss:.4f} auc={base_rate_auc:.4f}")

fold_log_losses = np.array([m["log_loss"] for m in per_fold_metrics])
fold_aucs = np.array([m["auc"] for m in per_fold_metrics])
overall_log_loss = log_loss(eval_df["is_correct"], eval_df["pred_llm"], labels=[0, 1])
overall_auc = roc_auc_score(eval_df["is_correct"], eval_df["pred_llm"])

print("=" * 60)
print(f"CV log loss: mean={fold_log_losses.mean():.4f} std={fold_log_losses.std():.4f}")
print(f"CV AUC:      mean={fold_aucs.mean():.4f} std={fold_aucs.std():.4f}")
print(f"Overall (stacked OOF, folds so far) log loss: {overall_log_loss:.4f}")
print(f"Overall (stacked OOF, folds so far) AUC:      {overall_auc:.4f}")
print("=" * 60)


In [ ]:
# --- Compare against Notebook B's baseline ------------------------------------
baseline_metrics_path = Path("/kaggle/input/trace-the-ace-baseline/baseline_metrics.json")
if not baseline_metrics_path.exists():
    # try the nested-mount layout, same quirk as the raw/features datasets
    nested = Path("/kaggle/input/datasets")
    if nested.exists():
        for owner_dir in nested.iterdir():
            candidate = owner_dir / "trace-the-ace-baseline" / "baseline_metrics.json"
            if candidate.exists():
                baseline_metrics_path = candidate
                break

if baseline_metrics_path.exists():
    baseline_metrics = json.load(open(baseline_metrics_path))
    print("Notebook B baseline (loaded from trace-the-ace-baseline):")
    print(f"  log loss: {baseline_metrics['oof_overall']['log_loss']:.4f}")
    print(f"  AUC:      {baseline_metrics['oof_overall']['auc']:.4f}")
else:
    print("trace-the-ace-baseline not attached to this notebook -- printing target "
          "numbers from Context.md instead:")
    print("  reference blog baseline: log loss 0.6054, AUC 0.5473")
    print("  current personal best:   log loss 0.5961, AUC 0.6430")

print(f"\nThis notebook (LLM, folds so far): log loss {overall_log_loss:.4f}, AUC {overall_auc:.4f}")


## 5. Outputs

Writes whatever has been completed so far to the final output filenames.
If fewer than 5 folds have been run, `oof_predictions_llm.csv` and
`llm_train_metrics.json` will only cover the completed folds -- re-run
section 3.5 with the remaining fold indices (across additional Kaggle
sessions) and re-run this section to refresh them once all 5 are done.

**Coverage caveat for Notebook D**: because of `VAL_SAMPLE_SIZE` (section
3.3), `oof_predictions_llm.csv` will NOT have a row for every `response_id`
in the full dataset, even once all 5 folds are done -- only a stratified
subsample of each fold's held-out responses were actually scored, to fit a
hard time budget. **Notebook D must `LEFT JOIN` on `response_id` and fall
back to `pred_baseline` (or another neutral value) wherever `pred_llm` is
missing**, rather than assuming full coverage. `llm_train_metrics.json`
records exactly how many responses were scored per fold (`n_val` vs.
`n_val_full`) so this gap is auditable.


In [ ]:
# --- oof_predictions_llm.csv (response_id, pred_llm) ---
oof_predictions_llm = eval_df[["response_id", "pred_llm"]].copy()
oof_predictions_llm.to_csv(OUT_DIR / "oof_predictions_llm.csv", index=False)
print(oof_predictions_llm.shape)
oof_predictions_llm.head()


In [ ]:
# --- llm_train_metrics.json (same format as baseline_metrics.json) ---
llm_metrics = {
    "model": "qwen2.5-7b-instruct-qlora-classifier" if loaded_model_name.startswith("Qwen")
             else "llama-3.1-8b-instruct-qlora-classifier",
    "base_model": loaded_model_name,
    "max_seq_len": MAX_SEQ_LEN,
    "lora_config": LORA_CONFIG,
    "n_folds": N_FOLDS,
    "folds_completed": folds_done,
    "per_fold": per_fold_metrics,
    "cv_mean": {"log_loss": float(fold_log_losses.mean()), "auc": float(fold_aucs.mean())},
    "cv_std": {"log_loss": float(fold_log_losses.std()), "auc": float(fold_aucs.std())},
    "oof_overall": {"log_loss": float(overall_log_loss), "auc": float(overall_auc)},
}

with open(OUT_DIR / "llm_train_metrics.json", "w") as f:
    json.dump(llm_metrics, f, indent=2)

print(json.dumps({k: v for k, v in llm_metrics.items() if k != "per_fold"}, indent=2))


In [ ]:
# --- Confirm lora_adapter/fold_i/ directories present for whichever folds ran ---
for p in sorted(ADAPTER_ROOT.glob("fold_*")):
    size_mb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
    print(f"{p}: {size_mb:.1f} MB")

for fname in ["oof_predictions_llm.csv", "llm_train_metrics.json"]:
    p = OUT_DIR / fname
    print(f"{fname}: {p.stat().st_size / 1e3:.1f} KB")
